# Information Theory Diagnostics for Cellular Infrastructure

This notebook computes and visualizes information theory metrics on cellular telemetry from Airtel Rwanda. We investigate:
1. **Shannon Entropy** of diurnal traffic volumes to analyze network demand predictability.
2. **Kullback-Leibler (KL) Divergence** to compare traffic profile variations between weekdays and weekends.
3. **Mutual Information** between Radio Access Technology (RAT) generations and traffic types.
4. **RRC State space & Feedback States** estimation.
5. **Satellite RTT Wall & Channel Dispersion** calculation.
6. **Peak Age of Information (AoI) Optimization** over core signaling queues.
7. **Zipf-Mandelbrot Content Entropy & Edge Caching Limits** calculation.
8. **Effective Capacity** under delay constraints.
9. **Transfer Entropy (Causality)** from DNS requests to GTP-C sessions.
10. **TCP Congestion Window KL Divergence** to BDP targets.
11. **Hardware-Conditional Entropy** of signaling overhead by TAC class.
12. **Age of Incorrect Information (AoII)** optimization over CTMC RRC states.
13. **Goal-Oriented Private Semantic Caching** Pareto frontier optimization.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import fynesse
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 120

## 1. Data Ingestion
We load the aggregated temporal trace of TCP/UDP flows.

In [ ]:
df = fynesse.load_joined_temporal_data()
df.head()

## 2. Diurnal Demand Predictability (Shannon Entropy)
We compute the Shannon Entropy $H(X)$ of hourly traffic volume distribution over the day:
$$H(X) = - \sum_{h=0}^{23} p_h \log_2 p_h$$

In [ ]:
entropy = fynesse.calculate_hourly_traffic_entropy(df)

## 3. Weekday vs. Weekend Diurnal Divergence (Kullback-Leibler Divergence)
We measure the relative entropy difference between weekday hourly traffic distribution ($P$) and weekend hourly traffic distribution ($Q$):
$$D_{\text{KL}}(P \parallel Q) = \sum_{h=0}^{23} P_h \log_2 \left(\frac{P_h}{Q_h}\right)$$

In [ ]:
kl_div = fynesse.calculate_weekday_weekend_kl_divergence(df)

## 4. RAT vs. Traffic Code (Mutual Information)
We evaluate how much information the choice of Radio Access Technology (RAT) shares with the type of traffic (uplink vs. downlink, payload vs. signaling):
$$I(RAT; Code) = \sum_{x \in RAT} \sum_{y \in Code} p(x,y) \log_2 \left(\frac{p(x,y)}{p(x)p(y)}\right)$$

In [ ]:
mutual_info = fynesse.calculate_rat_traffic_code_mutual_information(df)

## 5. RRC State space & Feedback States
We estimate the stationary distribution vector $\mathbf{P}_S$ of Dedicated, Shared, and Idle states:
$$\mathbf{P}_S = [\pi_D, \pi_S, \pi_I]$$

In [ ]:
qos_df = fynesse.load_qos_metrics()
p_s = fynesse.estimate_feedback_channel_states(qos_df)

## 6. Satellite RTT Wall & Channel Dispersion
We extract the RTT step-function above 200ms and compute the empirical channel dispersion $V_{\text{sat}}$ in seconds squared:
$$V_{\text{sat}} = \mathbb{E}[\text{Var}[\text{RTT} \mid X]]$$

In [ ]:
wan_df = fynesse.load_rtt_data(sheet_name="wan")
mean_rtt, var_rtt = fynesse.estimate_satellite_dispersion(wan_df)

## 7. Peak Age of Information (AoI) Optimization
We process core GTP-C request timelines to calculate arrival rate and solve the $M/GI/1/K$ queueing-theoretic model to locate the optimal arrival rate $\lambda^*$ that minimizes information staleness:
$$\mathbb{E}[\Delta_{\text{peak}}] = \mathbb{E}[T] + \frac{1}{\lambda_{\text{sig}} (1 - P_{\text{drop}})}$$

In [ ]:
gtpc_df = fynesse.load_gtpc_signaling()
opt_lambda, opt_aoi = fynesse.estimate_core_signaling_aoi(gtpc_df)

## 8. Zipf-Mandelbrot Content Entropy & Edge Caching Limits
We model domain requests using the Zipf-Mandelbrot distribution:
$$P(r) = \frac{C}{(r + q)^\alpha}$$
We calculate the content request entropy:
$$H(\text{Content}) = - \sum_{r=1}^{N} P(r) \log_2 P(r)$$
And compute the maximum theoretical cache hit rate for a cache size of $K$ elements:
$$\eta(K) = \sum_{r=1}^{K} P(r)$$

In [ ]:
entropy_c, eta10, eta50 = fynesse.calculate_zipf_mandelbrot_caching()

## 9. Effective Capacity under delay constraints
We evaluate the Effective Capacity $E_c(\theta)$ showing the maximum constant arrival rate supported under a statistical delay-bound QoS exponent $\theta$:
$$E_c(\theta) = - \frac{1}{\theta} \lim_{t \to \infty} \frac{1}{t} \ln \mathbb{E}\left[e^{-\theta S(t)}\right]$$

In [ ]:
thetas, ec_2g, ec_3g = fynesse.calculate_effective_capacity()

## 10. DNS to GTP-C Transfer Entropy causality
We compute the Transfer Entropy $T_{X \to Y}$ (from DNS query timeline $X_t$ to GTP-C requests timeline $Y_t$) to mathematically prove causality from DNS resolution to core session storms:
$$T_{X \to Y} = \sum_{t=1}^N H(Y_t \mid Y_1^{t-1}) - H(Y_t \mid Y_1^{t-1}, X_1^{t-1})$$

In [ ]:
t_xy = fynesse.calculate_dns_gtpc_transfer_entropy()

## 11. TCP window BDP starvation KL divergence
We compute the relative entropy $D_{\text{KL}}(P_{\text{win}} \parallel P_{\text{BDP}})$ to prove window starvation on resource-constrained cellular downlinks:
$$D_{\text{KL}}(P_{\text{win}} \parallel P_{\text{BDP}}) = \sum_{w} P_{\text{win}}(w) \log_2 \left( \frac{P_{\text{win}}(w)}{P_{\text{BDP}}(w)} \right)$$

In [ ]:
kl_win = fynesse.calculate_tcp_window_bdp_starvation()

## 12. Hardware-Conditional Entropy of Signaling Overhead
We calculate the conditional entropy $H(S \mid D)$ of signaling overhead ratio $S$ given the hardware class $D$:
$$H(S \mid D) = \sum_{d \in \mathcal{D}} P(D=d) H(S \mid D=d)$$

In [ ]:
h_s, h_s_d = fynesse.calculate_hardware_conditional_entropy()

## 13. Age of Incorrect Information (AoII) CTMC Optimization
We model RRC state transitions as a Continuous-Time Markov Chain (CTMC) and optimize the update threshold $\tau$ to minimize the Age of Incorrect Information (AoII) at the core gateway:
$$A(t) = (t - t_0) \cdot \mathbb{I}(S(t) \neq \hat{S}(t))$$
The optimization problem is formulated as:
$$\min_{\tau > 0} \mathbb{E}[A(\tau)] \quad \text{s.t.} \quad \lambda_{\text{update}}(\tau) \leq \Lambda_{\text{max}}$$

In [ ]:
opt_tau, opt_aoii, opt_rate = fynesse.calculate_aoii_rrc_optimization()

## 14. Private Semantic Caching Pareto Frontier Optimization
We optimize local edge caching task completion utility under user privacy constraints, tracing the Pareto frontier between information leakage $\epsilon$ and semantic utility:
$$\max_{x} \sum_{f} P(f) V_f x_f \quad \text{s.t.} \quad I(X_{\text{cache}}; D) \leq \epsilon$$

In [ ]:
eps_range, utility_vals = fynesse.calculate_private_semantic_caching()